[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/llama_index_sql_retriever_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">SQL 路由查询引擎示例</h1>

LlamaIndex 提供高级 API，几行代码即可构建强大的应用，但理解内部执行过程和定位问题往往并不容易。Phoenix 将查询引擎每次调用的底层结构可视化，并根据延迟、token 数及其他评估指标展示存在问题的 `span`，让 LLM 应用具备*可观测性*。

本教程将：
- 使用 LlamaIndex 构建同时使用 SQL 检索器和 VectorStoreIndex 的查询引擎
- 使用全局 `arize_phoenix` 处理器，以 [OpenInference 追踪](https://github.com/Arize-ai/openinference)格式记录数据
- 观察较复杂的 LlamaIndex 应用如何执行检索

ℹ️ 本 Notebook 需要 OpenAI API 密钥。

## 1. 安装依赖并导入库

安装 Phoenix、LlamaIndex 和 OpenAI。

In [ ]:
!pip install "arize-phoenix[evals,llama-index]" "openai>=1" gcsfs nest-asyncio "llama-index-readers-wikipedia" "sqlalchemy" wikipedia

In [ ]:
import os
from getpass import getpass

import openai
import pandas as pd
import wikipedia
from llama_index.core import Document, Settings
from llama_index.core.indices import VectorStoreIndex
from llama_index.core.query_engine import NLSQLTableQueryEngine, RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool
from llama_index.core.utilities.sql_wrapper import SQLDatabase
from llama_index.llms.openai import OpenAI
from sqlalchemy import (
    create_engine,
    text,
)

import phoenix as px

pd.set_option("display.max_colwidth", 1000)

## 2. 启动 Phoenix

可以在后台运行 Phoenix，收集已通过 `OpenInferenceTraceCallbackHandler` 添加埋点的 LlamaIndex 应用发出的追踪数据。Phoenix 支持 LlamaIndex 的[一键可观测性](https://gpt-index.readthedocs.io/en/latest/end_to_end_tutorials/one_click_observability.html)，可自动为应用添加埋点！详细说明见[集成指南](https://arize.com/docs/phoenix/integrations/llamaindex)。

启动 Phoenix，按照单元输出中的说明打开 Phoenix 界面。由于尚未运行应用，此时界面应为空。

In [ ]:
(session := px.launch_app()).view()

通过 `LlamaIndexInstrumentor` 启用 Phoenix 追踪。Phoenix 使用 OpenInference 追踪：这是一种用于捕获和存储 LLM 应用追踪的开源标准，使应用能够无缝接入 Phoenix 等 LLM 可观测性解决方案。

In [ ]:
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from phoenix.otel import register

tracer_provider = register(endpoint="http://127.0.0.1:6006/v1/traces")
LlamaIndexInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)

## 3. 配置 OpenAI API 密钥

如果尚未通过环境变量设置 OpenAI API 密钥，请在此设置。

In [ ]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")
openai.api_key = openai_api_key
os.environ["OPENAI_API_KEY"] = openai_api_key

## 3. 准备参考数据

首先下载包含多种数码相机技术参数的数据集，并将其转换为内存 SQL 数据库。数据集由 Kaggle 提供，详情见[此处](https://www.kaggle.com/datasets/crawford/1000-cameras-dataset)。

In [ ]:
camera_info = pd.read_parquet(
    "https://storage.googleapis.com/arize-phoenix-assets/datasets/structured/camera-info/cameras.parquet"
)

In [ ]:
camera_info.head()

In [ ]:
engine = create_engine("sqlite:///:memory:", future=True)
camera_info.to_sql("cameras", engine, index=False)

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT * FROM cameras LIMIT 5")).all()

    for row in result:
        print(row)

针对数码相机的通用问题，使用 `wikipedia` SDK 下载维基百科的数码相机页面，并将该文档转换为 LlamaIndex `VectorStoreIndex`。

In [ ]:
# 加载维基百科的 Digital Camera 页面
page = wikipedia.page(pageid=52797)
doc = Document(id_=page.pageid, text=page.content)

vector_indices = []
vector_index = VectorStoreIndex.from_documents([doc])
vector_indices.append(vector_index)

## 4. 构建 LlamaIndex 应用

使用包含多个查询引擎工具的简单 `RouterQueryEngine`，将查询路由到 SQL 检索器或基于维基百科“Digital Camera”页面构建的向量索引。

In [ ]:
Settings.llm = OpenAI(temperature=0.0, model="gpt-4o")

In [ ]:
sql_database = SQLDatabase(engine, include_tables=["cameras"])

sql_query_engine = NLSQLTableQueryEngine(
    sql_database=sql_database,
    tables=["cameras"],
)
sql_tool = QueryEngineTool.from_defaults(
    query_engine=sql_query_engine,
    description=(
        "用于将自然语言查询转换为 SQL，以查询"
        "包含具体数码相机技术细节的表，其列包括：Model,"
        " Release date, Max resolution, Low resolution, Effective pixels, Zoom wide (W),"
        " Zoom tele (T), Normal focus range, Macro focus range, Storage included,"
        " Weight (inc. batteries), Dimensions, Price"
    ),
)

vector_query_engines = [index.as_query_engine() for index in vector_indices]
vector_tools = []
for query_engine in vector_query_engines:
    vector_tool = QueryEngineTool.from_defaults(
        query_engine=query_engine,
        description="用于回答数码相机的一般性问题。",
    )
    vector_tools.append(vector_tool)

In [ ]:
query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=([sql_tool] + vector_tools),
)

## 5. 发起查询并在 Phoenix 中查看 span

In [ ]:
response = query_engine.query("What is the most expensive digital camera?")
print(str(response))

此查询询问某台相机的具体参数，因此被路由到 SQL 检索器，以获取回答所需的上下文。可以在 Phoenix 的 span 中查看 LLM 生成的 SQL。

![Phoenix 界面中的 SQL 检索](https://storage.googleapis.com/arize-phoenix-assets/assets/docs/notebooks/tracing/llama-index-sql-retrieval-tutorial/sql-retrieval.png)

In [ ]:
response = query_engine.query("Tell me about the history of digital camera sensors.")
print(str(response))

更通用的查询会被路由到向量索引。

![Phoenix 界面中的向量检索](https://storage.googleapis.com/arize-phoenix-assets/assets/docs/notebooks/tracing/llama-index-sql-retrieval-tutorial/vectorstoreindex-retrieval.png)

## 6. 总结

LLM 追踪及配套的 OpenInference Tracing 规范用于描述一种遥测数据，帮助理解 LLM 执行过程及周边应用上下文。对于组合 SQL 检索器和常见向量索引等多种检索策略的复杂 RAG 应用，这尤其有用。

有关 Phoenix、LLM 追踪和 LLM 评估的更多信息，请参阅[文档](https://arize.com/docs/phoenix/)。